# Part 3 — μP Training on Colab (Small → Medium → Large → XL)

**CS-GY 6923 · NYU Tandon · Spring 2026**

This notebook trains the 4 remaining μP model sizes (Small ~3M, Medium ~11M, Large ~33M, XL ~87M)
using the transferred LR of **1e-2** found in the Tiny sweep (Task 3.3).

---

## ✅ Pre-flight checklist — do this on your LOCAL machine BEFORE opening Colab

### Step 1 — Push latest code to GitHub
```bash
cd ~/Desktop/ML-Final-Project
git add scripts/mup_model.py scripts/mup_train.py scripts/mup_lr_sweep.py \
        scripts/mup_train_all.py scripts/compare_scaling.py \
        configs/ notebooks/ requirements.txt
git commit -m 'Add Part 3: μP model, training scripts, LR sweep notebook'
git push origin main
```

### Step 2 — Upload tokenized data to Google Drive
Upload the following files to a folder called **`ML-Final-Project/`** inside *My Drive*:

| Local path | Drive path |
|------------|------------|
| `data/tokenized/train.npy` | `ML-Final-Project/data/tokenized/train.npy` |
| `data/tokenized/val.npy` | `ML-Final-Project/data/tokenized/val.npy` |
| `data/tokenized/test.npy` | `ML-Final-Project/data/tokenized/test.npy` |
| `tokenizer/` (whole folder) | `ML-Final-Project/tokenizer/` |

**File sizes:** train.npy ≈ 194 MB, val.npy + test.npy ≈ 4 MB, tokenizer/ ≈ 10 MB.
Use the **Drive desktop app** or drag-and-drop in the browser.

### Step 3 — Choose runtime
`Runtime → Change runtime type → A100 GPU` (Colab Pro required for A100).

### Step 4 — Run all cells in order (Shift+Enter or Runtime → Run all)

---
## ✅ After Colab finishes — sync results back to local
Download the generated `outputs/` folder from Drive to your local machine:
- **Browser:** Open drive.google.com → `ML-Final-Project/outputs/` → right-click → Download
- **Desktop app:** It auto-syncs if you have Google Drive for Desktop installed
- Merge the downloaded `outputs/` into your local `ML-Final-Project/outputs/`

Then locally run:
```bash
python3 scripts/compare_scaling.py   # generates SP vs μP plots
```

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. GPU check ──────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), '❌ No GPU — Runtime → Change runtime type → GPU (A100)'
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU : {gpu}')
print(f'VRAM: {vram:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

# Use bfloat16 on A100/H100 for 2× throughput; fall back to float32 otherwise
USE_BF16 = 'A100' in gpu or 'H100' in gpu or 'A6000' in gpu
DTYPE    = torch.bfloat16 if USE_BF16 else torch.float32
print(f'Mixed precision: {"bfloat16" if USE_BF16 else "float32 (no AMP)"}')

In [ ]:
# ── 3. Install mup + clone repo ───────────────────────────────────────────────
import subprocess, sys

# Install mup (not pre-installed in Colab)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mup', '-q'], check=True)
print('mup installed ✓')

# Clone the project repo (public GitHub)
import os
REPO_URL  = 'https://github.com/shubham739/ML-Final-Project.git'
REPO_DIR  = '/content/ML-Final-Project'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print('Repo cloned ✓')

# Add to Python path so we can import scripts/
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify mup imports
from mup import MuReadout, set_base_shapes, MuAdamW
print('mup imports OK ✓')

In [ ]:
# ── 4. Configure paths ────────────────────────────────────────────────────────
# ⚠️  EDIT DRIVE_ROOT if you put the folder somewhere else in your Drive
DRIVE_ROOT = '/content/drive/MyDrive/ML-Final-Project'

DATA_DIR   = f'{DRIVE_ROOT}/data/tokenized'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
CONFIG_DIR = f'{REPO_DIR}/configs'

# Create output dirs
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs',        exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/results',     exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/plots',       exist_ok=True)

# Verify data exists
for f in ['train.npy', 'val.npy']:
    p = f'{DATA_DIR}/{f}'
    assert os.path.exists(p), f'❌ Missing: {p}\nUpload data/tokenized/ to Drive first!'
    size_mb = os.path.getsize(p) / 1024**2
    print(f'  {f}: {size_mb:.0f} MB ✓')

print(f'\nDrive root : {DRIVE_ROOT}')
print(f'Data dir   : {DATA_DIR}')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
# ── 5. Shared training infrastructure ────────────────────────────────────────
# All the helpers needed for μP training — mirrors scripts/mup_train.py

import json, math, time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from mup import MuReadout, set_base_shapes, MuAdamW

# -- Model config --
from scripts.model    import ModelConfig
from scripts.mup_model import MupGPT, build_mup_model

device = torch.device('cuda')

# -- LR schedule (cosine with warmup) --
def get_lr(step, warmup_steps, total_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * step / max(warmup_steps, 1)
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * progress)) * (max_lr - min_lr)

# -- Data loading --
print('Loading data into RAM...')
train_data = np.array(np.memmap(f'{DATA_DIR}/train.npy', dtype='uint16', mode='r'))
val_data   = np.array(np.memmap(f'{DATA_DIR}/val.npy',   dtype='uint16', mode='r'))
print(f'Train: {len(train_data):,} tokens  |  Val: {len(val_data):,} tokens')

def get_batch(data, block_size, batch_size):
    ix = np.random.randint(0, len(data) - block_size, size=(batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def compute_val_loss(model, block_size, max_batches=None):
    model.eval()
    losses, step = [], 0
    for i in range(0, len(val_data) - block_size, block_size * 8):
        starts = list(range(i, min(i + block_size*8, len(val_data)-block_size), block_size))
        if not starts: break
        x = torch.stack([torch.from_numpy(val_data[s:s+block_size].astype(np.int64)) for s in starts]).to(device)
        y = torch.stack([torch.from_numpy(val_data[s+1:s+block_size+1].astype(np.int64)) for s in starts]).to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
        step += 1
        if max_batches and step >= max_batches: break
    model.train()
    return float(np.mean(losses)) if losses else float('nan')

def upsert_results(path, entry):
    results = []
    if os.path.exists(path):
        with open(path) as f: results = json.load(f)
    results = [r for r in results if r.get('name') != entry['name']]
    results.append(entry)
    results.sort(key=lambda r: r.get('param_count', 0))
    with open(path, 'w') as f: json.dump(results, f, indent=2)

print('\nInfrastructure ready ✓')

In [ ]:
# ── 6. Core training function ─────────────────────────────────────────────────

def train_mup(
    config_path,
    lr           = 1e-2,        # μP transferred LR (found on Tiny, Task 3.3)
    batch_size   = 16,          # larger physical batch for A100 (vs 4 locally)
    eff_tokens   = 65536,       # keep same effective batch as local
    weight_decay = 0.1,
    warmup_frac  = 0.05,
    grad_clip    = 1.0,
    eval_interval = 100,
    seed         = 42,
    skip_existing = True,       # skip if checkpoint already saved to Drive
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    with open(config_path) as f: cfg_dict = json.load(f)
    model_name = cfg_dict.get('name', os.path.splitext(os.path.basename(config_path))[0])
    mup_name   = f'{model_name}_mup'
    ckpt_dir   = f'{OUTPUT_DIR}/checkpoints/{mup_name}'
    ckpt_path  = f'{ckpt_dir}/checkpoint_final.pt'

    if skip_existing and os.path.exists(ckpt_path):
        print(f'\n[SKIP] {mup_name} — checkpoint already at {ckpt_path}')
        with open(f'{OUTPUT_DIR}/logs/{mup_name}_lr1e-2.json') as f:
            r = json.load(f)
        print(f'  val_loss = {r["val_loss"]:.4f}')
        return r

    config = ModelConfig.from_dict(cfg_dict)
    model  = build_mup_model(config).to(device)
    n_params = model.count_parameters()

    block_size       = config.block_size
    grad_accum       = max(1, eff_tokens // (batch_size * block_size))
    tokens_per_step  = batch_size * block_size * grad_accum
    total_steps      = math.ceil(len(train_data) / tokens_per_step)
    warmup_steps     = max(1, int(total_steps * warmup_frac))
    min_lr           = lr * 0.1

    print(f'\n{"="*60}')
    print(f'Training (μP): {mup_name}  |  lr={lr:.1e}  |  device={device}')
    print(f'  Params: {n_params:,} ({n_params/1e6:.1f}M)  |  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Steps: {total_steps:,}  |  Warmup: {warmup_steps}')
    print(f'  Eff batch: {tokens_per_step:,} tokens (phys_bs={batch_size}, accum={grad_accum})')
    print(f'{"="*60}')

    optimizer = MuAdamW(model.parameters(), lr=lr, betas=(0.9, 0.95),
                        eps=1e-8, weight_decay=weight_decay)
    scaler    = torch.amp.GradScaler('cuda', enabled=USE_BF16)

    os.makedirs(ckpt_dir, exist_ok=True)
    train_losses, tokens_seen, peak_mem = [], 0, 0.0
    t_start = t_last = time.time()

    model.train()
    for step in range(total_steps):
        cur_lr = get_lr(step, warmup_steps, total_steps, lr, min_lr)
        for pg in optimizer.param_groups: pg['lr'] = cur_lr

        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(grad_accum):
            x, y = get_batch(train_data, block_size, batch_size)
            with torch.amp.autocast('cuda', dtype=DTYPE, enabled=USE_BF16):
                _, loss = model(x, y)
            scaler.scale(loss / grad_accum).backward()
            step_loss += (loss / grad_accum).item()

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()

        tokens_seen += tokens_per_step
        mem = torch.cuda.max_memory_allocated() / 1024**2
        if mem > peak_mem: peak_mem = mem

        if step % eval_interval == 0 or step == total_steps - 1:
            elapsed = time.time() - t_start
            tps     = tokens_seen / elapsed if elapsed > 0 else 0
            val_est = compute_val_loss(model, block_size, max_batches=50)
            train_losses.append({'step': step, 'train_loss': round(step_loss, 5),
                                 'val_loss': round(val_est, 5), 'lr': round(cur_lr, 8),
                                 'tokens_per_sec': round(tps, 1)})
            print(f'  step {step:5d}/{total_steps} ({100*step/total_steps:4.1f}%)  '
                  f'train={step_loss:.4f}  val={val_est:.4f}  '
                  f'lr={cur_lr:.2e}  {tps:,.0f} tok/s  mem={peak_mem:.0f}MB')

    print('Computing final val loss over full val set...')
    final_val = compute_val_loss(model, block_size)
    total_time = time.time() - t_start
    avg_tps    = tokens_seen / total_time
    print(f'\nFinal val loss : {final_val:.4f}')
    print(f'Total time     : {total_time/3600:.2f} hrs  |  {avg_tps:,.0f} tok/s')
    print(f'Peak memory    : {peak_mem:.0f} MB')

    # Save checkpoint to Drive
    torch.save({'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': cfg_dict, 'val_loss': final_val, 'lr': lr,
                'parameterization': 'mup'}, ckpt_path)
    print(f'Checkpoint → {ckpt_path}')

    result = {'name': mup_name, 'base_name': model_name, 'param_count': n_params,
              'val_loss': final_val, 'lr': lr,
              'd_model': config.d_model, 'n_layers': config.n_layers,
              'n_heads': config.n_heads, 'd_ff': config.d_ff,
              'train_losses': train_losses,
              'total_time_seconds': round(total_time, 1),
              'tokens_per_second': round(avg_tps, 1),
              'peak_memory_mb': round(peak_mem, 1),
              'parameterization': 'mup'}

    log_path = f'{OUTPUT_DIR}/logs/{mup_name}_lr1e-2.json'
    with open(log_path, 'w') as f: json.dump(result, f, indent=2)
    print(f'Log → {log_path}')

    res_path = f'{OUTPUT_DIR}/results/mup_scaling_results.json'
    upsert_results(res_path, {k: v for k, v in result.items() if k != 'train_losses'})
    print(f'Results → {res_path}')

    del model; torch.cuda.empty_cache()
    return result

print('train_mup() defined ✓')

---
## Task 3.4 — Train all 4 remaining μP model sizes

LR = **1e-2** (transferred from Tiny sweep — no retuning per size).

Expected times on A100 (40 GB VRAM):

| Model | Params | Est. time |
|-------|--------|-----------|
| Small  | ~3M   | ~20 min   |
| Medium | ~12M  | ~30 min   |
| Large  | ~33M  | ~45 min   |
| XL     | ~87M  | ~75 min   |

Each cell can be re-run independently (checkpoints are saved to Drive after each model).
If the session resets, re-run cells 1–6 then resume from the cell that was interrupted.

In [ ]:
# ── 7. Train Small (~3M params) ───────────────────────────────────────────────
result_small = train_mup(
    config_path   = f'{CONFIG_DIR}/small.json',
    lr            = 1e-2,
    batch_size    = 16,
    eval_interval = 100,
    skip_existing = True,
)
print(f'\nSmall μP done → val_loss = {result_small["val_loss"]:.4f}')

In [ ]:
# ── 8. Train Medium (~12M params) ─────────────────────────────────────────────
result_medium = train_mup(
    config_path   = f'{CONFIG_DIR}/medium.json',
    lr            = 1e-2,
    batch_size    = 16,
    eval_interval = 100,
    skip_existing = True,
)
print(f'\nMedium μP done → val_loss = {result_medium["val_loss"]:.4f}')

In [ ]:
# ── 9. Train Large (~33M params) ──────────────────────────────────────────────
result_large = train_mup(
    config_path   = f'{CONFIG_DIR}/large.json',
    lr            = 1e-2,
    batch_size    = 16,
    eval_interval = 100,
    skip_existing = True,
)
print(f'\nLarge μP done → val_loss = {result_large["val_loss"]:.4f}')

In [ ]:
# ── 10. Train XL (~87M params) ────────────────────────────────────────────────
result_xl = train_mup(
    config_path   = f'{CONFIG_DIR}/xl.json',
    lr            = 1e-2,
    batch_size    = 16,
    eval_interval = 100,
    skip_existing = True,
)
print(f'\nXL μP done → val_loss = {result_xl["val_loss"]:.4f}')

In [ ]:
# ── 11. Results summary ───────────────────────────────────────────────────────
import json, os

res_path = f'{OUTPUT_DIR}/results/mup_scaling_results.json'
if os.path.exists(res_path):
    with open(res_path) as f:
        mup_results = json.load(f)

    print('μP Scaling Results (all models trained so far):')
    print(f"{'Name':<14} {'Params':>10} {'d_model':>8} {'n_lay':>6} {'Val Loss':>10} {'Time(h)':>8} {'tok/s':>8}")
    print('-' * 70)
    for r in sorted(mup_results, key=lambda x: x['param_count']):
        t   = r.get('total_time_seconds', 0)
        tps = r.get('tokens_per_second', 0)
        print(f"{r['name']:<14} {r['param_count']:>10,} {r['d_model']:>8} "
              f"{r['n_layers']:>6} {r['val_loss']:>10.4f} {t/3600:>8.2f} {tps:>8,.0f}")
else:
    print('No results file found yet — run training cells above first.')

In [ ]:
# ── 12. Training curves plot ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import glob

log_files = sorted(glob.glob(f'{OUTPUT_DIR}/logs/*_mup_lr*.json'))
if not log_files:
    print('No μP logs found — run training cells first.')
else:
    colors = plt.cm.plasma(plt.np.linspace(0.1, 0.9, len(log_files)))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for path, color in zip(log_files, colors):
        with open(path) as f: log = json.load(f)
        entries = log.get('train_losses', [])
        if not entries: continue
        steps  = [e['step']       for e in entries]
        t_loss = [e['train_loss'] for e in entries]
        v_loss = [e['val_loss']   for e in entries]
        label  = log.get('name', path)
        axes[0].plot(steps, t_loss, label=label, color=color, linewidth=1.5)
        axes[1].plot(steps, v_loss, label=label, color=color, linewidth=1.5)

    for ax, title in zip(axes, ['Train Loss', 'Val Loss']):
        ax.set_xlabel('Step'); ax.set_ylabel('Loss')
        ax.set_title(f'μP — {title} vs Step (all sizes)', fontsize=11)
        ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)

    fig.suptitle('Task 3.4 — μP Training Curves (all model sizes)', fontsize=12)
    fig.tight_layout()
    fig.savefig(f'{OUTPUT_DIR}/plots/mup_training_curves.png', dpi=150)
    plt.show()
    print(f'Saved: {OUTPUT_DIR}/plots/mup_training_curves.png')

In [ ]:
# ── 13. Quick power-law fit on μP data (preview of Task 3.5) ─────────────────
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

res_path = f'{OUTPUT_DIR}/results/mup_scaling_results.json'
if not os.path.exists(res_path):
    print('No results yet — run training cells first.')
else:
    with open(res_path) as f: data = json.load(f)
    data.sort(key=lambda r: r['param_count'])
    params = np.array([r['param_count'] for r in data], dtype=float)
    losses = np.array([r['val_loss']    for r in data], dtype=float)

    def power_law(N, a, alpha, c): return a * N**(-alpha) + c

    if len(params) >= 3:
        try:
            popt, _ = curve_fit(power_law, params, losses,
                                p0=[1.0, 0.1, losses.min()*0.5],
                                bounds=([0,0,0],[np.inf,2,np.inf]), maxfev=30000)
            a, alpha, c = popt
            print(f'μP power-law fit: L = {a:.3f} · N^(-{alpha:.3f}) + {c:.3f}')
            print(f'  Scaling exponent α = {alpha:.3f}  (Kaplan NL: ~0.076)')

            N_fit = np.logspace(np.log10(params.min()*0.7), np.log10(params.max()*1.5), 300)
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.scatter(params, losses, s=80, color='darkorchid', zorder=5, label='μP models')
            for r in data:
                ax.annotate(r['name'], (r['param_count'], r['val_loss']),
                            textcoords='offset points', xytext=(6,4), fontsize=9)
            ax.plot(N_fit, power_law(N_fit, *popt), '--', color='darkorchid', linewidth=1.8,
                    label=rf'$L={a:.2f}\cdot N^{{-{alpha:.3f}}}+{c:.3f}$')
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlabel('Parameters (N)', fontsize=12)
            ax.set_ylabel('Validation Loss', fontsize=12)
            ax.set_title('μP Scaling Law (log-log) — Task 3.5 preview', fontsize=12)
            ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)
            fig.tight_layout()
            fig.savefig(f'{OUTPUT_DIR}/plots/mup_scaling_preview.png', dpi=150)
            plt.show()
        except Exception as e:
            print(f'Fit failed (need ≥3 models): {e}')
    else:
        print(f'Only {len(params)} models trained so far — need at least 3 for curve fit.')
        print(f'Models: {[r["name"] for r in data]}')
        print(f'Losses: {losses.tolist()}')

---
## ✅ Syncing results back to your local machine

After all 4 models finish, download the outputs from Drive and merge them locally.

### Option A — Google Drive Desktop App (easiest)
If you have **Google Drive for Desktop** installed, the `ML-Final-Project/outputs/` folder
auto-syncs to your Mac. Just wait a few minutes after training completes.

### Option B — Browser download
1. Open [drive.google.com](https://drive.google.com)
2. Navigate to `ML-Final-Project/outputs/`
3. Right-click → **Download** (creates a zip)
4. Unzip and merge into your local `outputs/` folder

### Option C — Run this cell to zip everything (then download from Colab files panel)

```python
import shutil
shutil.make_archive('/content/mup_outputs', 'zip', OUTPUT_DIR)
print('Download /content/mup_outputs.zip from the Files panel (left sidebar)')
```

### After syncing — run locally to generate comparison plots:

```bash
cd ~/Desktop/ML-Final-Project

# Merge the tiny_mup result (trained locally) into the file if needed
# (already there if you ran mup_train.py locally)

# Generate SP vs μP comparison plot + extrapolation
python3 scripts/compare_scaling.py

# Outputs:
#   outputs/plots/sp_vs_mup_scaling.png   ← log-log overlay
#   outputs/plots/lr_sweep_comparison.png  ← SP vs μP side-by-side
#   outputs/results/comparison_results.json
```

In [ ]:
# ── Optional: zip outputs for manual download ─────────────────────────────────
# Uncomment and run if you prefer downloading a zip from the Colab Files panel
# instead of using Drive.

# import shutil
# print('Zipping outputs...')
# shutil.make_archive('/content/mup_outputs', 'zip', OUTPUT_DIR)
# print('Done. Download /content/mup_outputs.zip from the Files panel (left sidebar →  folder icon).')